### Interactive Tokens and Embeddings Demonstrations for Machine Learning

Michael J. Pyrcz, Professor, The University of Texas at Austin 

[Twitter](https://twitter.com/geostatsguy) | [GitHub](https://github.com/GeostatsGuy) | [Website](http://michaelpyrcz.com) | [GoogleScholar](https://scholar.google.com/citations?user=QVZ20eQAAAAJ&hl=en&oi=ao) | [Geostatistics Book](https://www.amazon.com/Geostatistical-Reservoir-Modeling-Michael-Pyrcz/dp/0199731446) | [YouTube](https://www.youtube.com/channel/UCLqEr-xV-ceHdXXXrTId5ig)  | [Applied Geostats in Python e-book](https://geostatsguy.github.io/GeostatsPyDemos_Book/intro.html) | [Applied Machine Learning in Python e-book](https://geostatsguy.github.io/MachineLearningDemos_Book/) | [LinkedIn](https://www.linkedin.com/in/michael-pyrcz-61a648a1)

The goal of these interactive demonstrations is to explore fundamental concepts in representation learning and understand how data can be transformed into more useful representations for machine learning.

Representation learning helps extract meaningful information from raw data, reveal important patterns, and incorporate expert knowledge. We will explore two approaches through interactive examples.

#### Interactive Dashboards

1. **Tokenization** – Transform continuous well-log measurements into meaningful geological tokens by extracting information and incorporating expert knowledge.

2. **Embeddings** – Expand the original feature space through a quadratic basis transformation to reveal patterns and improve a classification task.

#### Overall Learning Objective

Machine learning depends on how we represent our data. Tokenization and embeddings help transform raw observations into more meaningful representations that expose patterns, incorporate domain knowledge, and improve downstream machine learning tasks.

**Understand the data. Transform the representation. Reveal the patterns!**

#### Import Required Packages

We will use several standard Python packages for numerical calculations, visualization, and interactive dashboards. These packages should already be installed with Anaconda 3.

In [4]:
import numpy as np                                            # arrays and matrix math
import matplotlib.pyplot as plt                               # for plotting
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch
from matplotlib.lines import Line2D
import ipywidgets as widgets                                  # interactivity
from ipywidgets import interact
from IPython.display import display, HTML
from ipywidgets import Layout

#### Tokenization: Turning Data into Useful Information

Tokenization transforms raw data into a more meaningful and useful representation. It extracts important patterns, organizes observations into interpretable categories, and incorporates expert knowledge into the data.

In subsurface applications, tokenization can turn continuous measurements, such as porosity logs, into geological concepts such as sand, shale, thick sand, channels, and reservoir intervals. Each level of tokenization adds interpretation and context, helping us move from raw measurements toward information that is more useful for geological understanding and decision-making.

The key insight: Tokenization is more than assigning labels. It is a way to extract information, introduce domain knowledge, and create representations that better support downstream machine learning tasks.

In [5]:
np.random.seed(42)                                            # generate a synthetic porosity well log

dz = 0.1                                                      # well log configuration
depth = np.arange(dz/2, 100.0, dz)
n = len(depth)
geological_intervals = [                                      # construct synthetic geological intervals (top depth, base depth, mean porosity)
    (0,  8,  0.025),                                          # Overbank
    (8,  17, 0.160),                                          # Thick sand
    (17, 24, 0.025),                                          # Overbank
    (24, 26, 0.145),                                          # Thin sand
    (26, 34, 0.025),                                          # Overbank
    (34, 49, 0.175),                                          # Thick sand
    (49, 60, 0.035),                                          # Overbank
    (60, 64, 0.145),                                          # Thick sand
    (64, 75, 0.025),                                          # Overbank
    (75, 91, 0.160),                                          # Thick sand
    (91, 100, 0.030)                                          # Overbank
]

porosity = np.zeros(n)                                        # populate porosity from geological intervals

for top, base, mean_phi in geological_intervals:
    mask = (depth >= top) & (depth < base)
    porosity[mask] = mean_phi

noise = np.convolve(np.random.normal(0, 0.012, n),np.ones(11) / 11,mode='same') # add spatially correlated variation and small-scale noise

noise += np.random.normal(0, 0.003, n)
porosity = np.clip(porosity + noise, 0.005, 0.28)

phi_threshold = 0.08                                          # define level 1: sand and shale
tokens_level1 = np.where(porosity > phi_threshold,'Sand','Shale')

minimum_sand_thickness = 5.0                                  # define Level 2: thick sand intervals
tokens_level2 = np.full(n, 'Shale', dtype='<U15')             # initialize every sample as shale
sand_mask = porosity > phi_threshold                          # identify contiguous runs above the porosity threshold

changes = np.diff(np.concatenate(([False], sand_mask, [False])).astype(int))

starts = np.where(changes == 1)[0]; ends = np.where(changes == -1)[0]

for start, end in zip(starts, ends):                          # assign each complete sand interval
    thickness = (end - start) * dz
    if thickness >= minimum_sand_thickness:
        tokens_level2[start:end] = 'Thick sand'
    else:
        tokens_level2[start:end] = 'Thin sand'

rolling_window_m = 3.0                                        # define Level 3: Channel and Overbank / a centered rolling average of porosity

window_samples = int(round(rolling_window_m / dz))    
pad_left = window_samples // 2                                # preserve the number of samples at the well boundaries
pad_right = window_samples - 1 - pad_left

porosity_padded = np.pad(porosity,(pad_left, pad_right),mode='edge')
rolling_average = np.convolve(porosity_padded,np.ones(window_samples) / window_samples,mode='valid')
channel_threshold = 0.10                                      # assign geological architecture tokens
tokens_level3 = np.where(rolling_average > channel_threshold,'Channel','Overbank')

colors_level1 = {'Sand': 'yellow','Shale': 'gray'}
colors_level2 = {'Thick sand': 'gold','Thin sand': 'lightsalmon','Shale': 'slategray'}
colors_level3 = {'Channel': 'goldenrod','Overbank': 'slategray'}
colors_level4 = {'Reservoir': 'seagreen','Non-reservoir': 'slategray'}

tokens_level4 = np.full(n, 'Reservoir', dtype='<U15')         # define Level 4: reservoir and non-reservoir

def plot_tokenization(level):                                 # dashboard plotting function
    if level == 1:                                            # select tokens and colors based on level
        tokens = tokens_level1
        color_map = colors_level1
        level_title = 'Level 1: Lithology'
        rule_text = r'Sand: $\phi > 0.04$ | Shale: $\phi \leq 0.04$'
        reference_threshold = phi_threshold
    elif level == 2:
        tokens = tokens_level2
        color_map = colors_level2
        level_title = 'Level 2: Sand Thickness'
        rule_text = 'Thick sand: continuous sand interval >= 3 m'
        reference_threshold = phi_threshold
    elif level == 3:
        tokens = tokens_level3
        color_map = colors_level3
        level_title = 'Level 3: Geological Architecture'
        rule_text = r'Channel: 3 m rolling average $\phi > 0.10$'
        reference_threshold = channel_threshold
    else:
        tokens = tokens_level4
        color_map = colors_level4
        level_title = 'Level 4: Reservoir Classification'
        rule_text = 'All observations classified as Reservoir'
        reference_threshold = 0.04
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 7),sharey=True) # create two side-by-side well log plots
    ax1.plot(porosity, depth,color='black', linewidth=1.2) # left: original continuous porosity well log
    ax1.fill_betweenx(depth, 0, porosity,color='lightgray', alpha=0.55)
    ax1.axvline(reference_threshold,color='red', linestyle='--',linewidth=1.5, alpha=0.8)
    ax1.set_xlabel('Porosity', fontsize=11); ax1.set_ylabel('Depth (m)', fontsize=11)
    ax1.set_title('Original Well Log',fontsize=13); ax1.set_xlim(0, 0.28); ax1.set_ylim(100, 0)
    ax1.set_xticks(np.arange(0, 0.301, 0.05)); ax1.grid(axis='both', alpha=0.25)

    for token, color in color_map.items():                    # right: tokenized well log
        mask = tokens == token
        ax2.scatter(porosity[mask],depth[mask],color=color,s=15,edgecolors='none',label=token,zorder=3)
    ax2.set_xlabel('Porosity', fontsize=11)
    ax2.set_title('Tokenized Well Log',fontsize=13)

    ax2.set_xlim(0, 0.28); ax2.set_ylim(100, 0); ax2.set_xticks(np.arange(0, 0.301, 0.05))
    ax2.grid(axis='both', alpha=0.25)

    legend_handles = [Line2D([0], [0],marker='o',color='none',markerfacecolor=color,
        markeredgecolor='none',markersize=8,label=token) for token, color in color_map.items()]
    ax2.legend(handles=legend_handles,loc='lower right',fontsize=9,frameon=True)
    fig.suptitle('Geological Tokenization of a Porosity Well Log', fontsize=15, y=0.98) # overall dashboard formatting
    fig.text(0.5, 0.935, rule_text, ha='center',fontsize=10 )
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show(); plt.close(fig)
    unique_tokens, counts = np.unique(tokens, return_counts=True) # display token summary
    summary = ' &nbsp; | &nbsp; '.join(
        [
            f'<b>{token}:</b> {count} samples '
            f'({100 * count / n:.1f}%)'
            for token, count in zip(unique_tokens, counts)
        ]
    )

    display(HTML(
        f"""
        <div style="text-align:center;
                    font-size:12px;
                    padding:8px;">
            <b>{level_title}</b><br>
            {summary}
        </div>
        """
    ))

level_slider = widgets.SelectionSlider(                       # create categorical level slider
    options=[('1: Sand/Shale', 1), ('2: Thick Sand', 2), ('3: Channel/Overbank', 3),
    ('4: Reservoir', 4)],value=1,description='Tokenization:',continuous_update=False,
    style={'description_width': 'initial'},layout=widgets.Layout(width='650px'))

dashboard = widgets.interactive_output(plot_tokenization,{'level': level_slider})  # Connect slider to dashboard

title = widgets.HTML(                                         # display dashboard
    """
    <h3 style="text-align:center;">
    From Continuous Well Logs to Geological Tokens
    </h3>
    <p style="text-align:center;">
    Explore three levels of geological tokenization
    using the same synthetic porosity well log.
    </p>
    """
)

display(widgets.VBox([title,widgets.HBox([level_slider],layout=widgets.Layout(justify_content='center')),
    dashboard],layout=widgets.Layout(align_items='center',width='100%')))

### Feature-Space Expansion and Representation Learning

Representation learning is the process of transforming data into a new feature space (embedding) where important patterns and relationships become easier to identify. By expanding the dimensionality of our data, we can reveal structure that may be difficult to recognize in the original feature space.

In this interactive demonstration, we start with two categories distributed along a single feature, $X_1$. We then introduce a quadratic basis expansion, adding a second feature, $X_2 = X_1^2$, to transform the observations onto a parabola. Use the slider to control the contribution of this new feature and observe how the representation changes.

The key insight: A classification problem that is not linearly separable in the original feature space can become linearly separable in the expanded feature space, allowing us to define a simple linear classification boundary.

Note: This example demonstrates engineered feature expansion (a polynomial basis transformation), a simple form of representation learning. Learned embeddings go a step further by learning the transformation from data.

In [6]:
x1_cat1 = np.array([-0.42, -0.30, -0.18, -0.08, 0.08,  0.18,  0.30,  0.42]) # category 1: observations near the origin
x1_cat0 = np.array([-1.80, -1.50, -1.25, -1.05, 1.05,  1.25,  1.50,  1.80]) # category 0: observations on both sides, away from the origin

x1 = np.concatenate([x1_cat1, x1_cat0])                       # combine observations and category labels
category = np.concatenate([np.ones(len(x1_cat1)),np.zeros(len(x1_cat0))])


color_cat1 = 'grey'; color_cat0 = 'gold'                      # fixed colors and markers
marker_cat1 = 'o'; marker_cat0 = 's'

x2_boundary = (np.max(x1_cat1**2) + np.min(x1_cat0**2)) / 2.0 # decision boundary: halfway between the closest categories

def plot_embedding(alpha):                                    # dashboard plotting function
    x2 = alpha * x1**2                                        # basis expansion

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))     # create two side-by-side plots

    ax1.scatter(x1_cat1, np.zeros_like(x1_cat1),color=color_cat1, marker=marker_cat1, # left: original feature space
        s=85, edgecolor='black', linewidth=0.7,label='Shale', zorder=3)

    ax1.scatter(x1_cat0, np.zeros_like(x1_cat0),color=color_cat0, marker=marker_cat0,
        s=85, edgecolor='black', linewidth=0.7,label='Sand', zorder=3)

    ax1.axhline(0, color='gray', linestyle='--',linewidth=1, alpha=0.6)

    ax1.set_xlim(-2.1, 2.1); ax1.set_ylim(-0.25, 3.7); ax1.set_xlabel(r'$X_1$', fontsize=12)
    ax1.set_ylabel(''); ax1.set_yticks([]); ax1.set_title('Original Feature Space', fontsize=13)

    ax1.grid(axis='x', alpha=0.25); ax1.legend(loc='upper right', fontsize=9) 

    ax2.scatter(x1_cat1, alpha * x1_cat1**2,color=color_cat1, marker=marker_cat1, # right: expanded feature space - cat. 1
        s=85, edgecolor='black', linewidth=0.7,label='Shale', zorder=3)

    # Category 0
    ax2.scatter(x1_cat0, alpha * x1_cat0**2,color=color_cat0, marker=marker_cat0,
        s=85, edgecolor='black', linewidth=0.7,label='Sand', zorder=3)

    if alpha > 0:                                             # linear decision boundary
        boundary = alpha * x2_boundary
        ax2.axhline(boundary, color='black',linestyle='--', linewidth=2,label='Linear decision boundary',zorder=2)

        ax2.text(0.97, boundary + 0.10,'Decision boundary',ha='right', va='bottom',fontsize=9, color='black')
    else:
        ax2.text(0.5, 0.90,'No separation possible',transform=ax2.transAxes,ha='center', fontsize=10,
            bbox=dict(facecolor='white',edgecolor='white', alpha=0.9))

    ax2.set_xlim(-2.1, 2.1); ax2.set_ylim(-0.25, 3.7); ax2.set_xlabel(r'$X_1$', fontsize=12)
    ax2.set_ylabel(r'$X_2$', fontsize=12); ax2.set_title('Expanded Feature Space', fontsize=13)

    ax2.grid(alpha=0.25); ax2.legend(loc='upper right', fontsize=9)

    fig.suptitle('Feature-Space Expansion and Linear Separability',fontsize=14, y=1.04) # dashboard formatting
    plt.tight_layout(); plt.close(fig); display(fig)          # display without retaining an additional open figure
    
    if alpha == 0:                                            # numerical dashboard information
        status = 'No separation: all observations have X2 = 0'
        gap = 0.0
    else:
        gap = alpha * (np.min(x1_cat0**2) - np.max(x1_cat1**2))
        status = 'Categories linearly separable'

    display(HTML(f"""<div style="text-align:center; padding:8px;font-size:13px;"><b>Embedding factor:</b> {alpha:.2f}
        &nbsp;&nbsp; | &nbsp;&nbsp; <b>Vertical separation:</b> {gap:.3f} &nbsp;&nbsp; | &nbsp;&nbsp; <b>Status:</b> {status}</div> """))

alpha_slider = widgets.FloatSlider(value=0.0,min=0.0,max=1.0,step=0.01,description='Embedding factor:', # interactive slider
    continuous_update=False,readout_format='.2f',style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px'))

dashboard = widgets.interactive_output(plot_embedding,{'alpha': alpha_slider}) # interactive output

title = widgets.HTML(                                         # dashboard title and display
    """
    <h3 style="text-align:center;">
    Embeddings Demonstration, Prof. Michael Pyrcz, The University of Texas at Austin
    </h3>
    <p style="text-align:center;">
    Explore how a quadratic basis expansion transforms
    observations into a linearly separable representation.
    </p>
    """
)

controls = widgets.HBox([alpha_slider],layout=widgets.Layout(justify_content='center',align_items='center',gap='30px')) # stack widgets
display(widgets.VBox([title, controls, dashboard],layout=widgets.Layout(align_items='center',width='100%'))) # instatniate display

#### Comments

This was a simple demonstration of toekenization and embedding in machine learning with interactive plots in Jupyter Notebook Python with the ipywidgets and matplotlib packages. 

I have many other demonstrations on data analytics and machine learning, e.g. on the basics of working with DataFrames, ndarrays, univariate statistics, plotting data, declustering, data transformations, trend modeling and many other workflows available at https://github.com/GeostatsGuy/PythonNumericalDemos and https://github.com/GeostatsGuy/GeostatsPy. 
  
I hope this was helpful,

*Michael*

#### About the Author

Michael Pyrcz is a professor in the [Cockrell School of Engineering](https://cockrell.utexas.edu/faculty-directory/alphabetical/p), and the [Jackson School of Geosciences](https://www.jsg.utexas.edu/researcher/michael_pyrcz/), at [The University of Texas at Austin](https://www.utexas.edu/), where he researches and teaches subsurface, spatial [data analytics](MachineLearning_glossary_d_f.ipynb#data-analytics), [geostatistics](MachineLearning_glossary_g_l.ipynb#geostatistics), and machine learning. Michael is also,

* the principal investigator of the [Energy Analytics](https://fri.cns.utexas.edu/energy-analytics) freshmen research initiative and a [core](MachineLearning_glossary_a_c.ipynb#core) faculty in the Machine Learn Laboratory in the College of Natural Sciences, The University of Texas at Austin

* an associate editor for [Computers and Geosciences](https://www.sciencedirect.com/journal/computers-and-geosciences/about/editorial-board), and a board member for [Mathematical Geosciences](https://link.springer.com/journal/11004/editorial-board), the International Association for Mathematical Geosciences. 

Michael has written over 100 [peer-reviewed publications](https://scholar.google.com/citations?user=QVZ20eQAAAAJ&hl=en), a [Python package](https://pypi.org/project/geostatspy/) for spatial [data](MachineLearning_glossary_d_f.ipynb#data) analytics, co-authored a textbook on spatial data analytics, [Geostatistical Reservoir Modeling](https://www.amazon.com/Geostatistical-Reservoir-Modeling-Michael-Pyrcz/dp/0199731446) and author of two recently released e-books, [Applied Geostatistics in Python: a Hands-on Guide with GeostatsPy](https://geostatsguy.github.io/GeostatsPyDemos_Book/intro.html) and [Applied Machine Learning in Python: a Hands-on Guide with Code](https://geostatsguy.github.io/MachineLearningDemos_Book/intro.html).

All of Michael’s university lectures are available on his [YouTube Channel](https://www.youtube.com/@GeostatsGuyLectures) with links to 100s of Python interactive dashboards and well-documented workflows in over 40 repositories on his [GitHub account](https://github.com/GeostatsGuy), to support any interested students and working professionals with evergreen content. To find out more about Michael’s work and shared educational resources visit his [Website](https://www.michaelpyrcz.com).

#### Want to Work Together?

I hope this content is helpful to those that want to learn more about subsurface modeling, [data analytics](MachineLearning_glossary_d_f.ipynb#data-analytics) and machine learning. Students and working professionals are welcome to participate.

* Want to invite me to visit your company for training, mentoring, project review, workflow design and / or consulting? I'd be happy to drop by and work with you! 

* Interested in partnering, supporting my graduate student research or my Subsurface [Data](MachineLearning_glossary_d_f.ipynb#data) Analytics and Machine Learning consortium (co-PI is Professor John Foster)? My research combines data analytics, stochastic modeling and machine learning theory with practice to develop novel methods and workflows to add [value](MachineLearning_glossary_s_z.ipynb#value). We are solving challenging subsurface problems!

* I can be reached at mpyrcz@austin.utexas.edu.

I'm always happy to discuss,

*Michael*

Michael Pyrcz, Ph.D., P.Eng. Professor, Cockrell School of Engineering and The Jackson School of Geosciences, The University of Texas at Austin

More Resources Available at: [Twitter](https://twitter.com/geostatsguy) | [GitHub](https://github.com/GeostatsGuy) | [Website](http://michaelpyrcz.com) | [GoogleScholar](https://scholar.google.com/citations?user=QVZ20eQAAAAJ&hl=en&oi=ao) | [Geostatistics Book](https://www.amazon.com/Geostatistical-Reservoir-Modeling-Michael-Pyrcz/dp/0199731446) | [YouTube](https://www.youtube.com/channel/UCLqEr-xV-ceHdXXXrTId5ig)  | [Applied Geostats in Python e-book](https://geostatsguy.github.io/GeostatsPyDemos_Book/intro.html) | [Applied Machine Learning in Python e-book](https://geostatsguy.github.io/MachineLearningDemos_Book/) | [LinkedIn](https://www.linkedin.com/in/michael-pyrcz-61a648a1)